# 02 — Cross-lingual Transfer Experiments (Paper 1)

Reproduce the zero-shot cross-lingual transfer benchmark from arXiv:2409.10965.

**Task**: Train on Kinyarwanda news, evaluate on Kirundi news (14 categories).

**Models**: mBERT, AfriBERTa, BantuBERTa (transformers) + BiGRU, TextCNN, CharCNN (traditional)

**Novel metric**: Catastrophic forgetting % — source accuracy drop after target fine-tuning.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mrm.data.datasets import HFTextDataset, SimpleVocabDataset
from mrm.models.transformer_clf import load_transformer_classifier, make_training_args, SUPPORTED_MODELS
from mrm.models.bigru import BiGRUClassifier
from mrm.training.trainer import build_hf_trainer, compute_metrics
from mrm.training.callbacks import ForgettingMonitor
from mrm.evaluation.evaluate import evaluate_crosslingual
from mrm.evaluation.metrics import compute_forgetting_percentage

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

## 1. Load Data

In [ ]:
MODEL_NAME = 'afriberta'  # change to mbert or bantubert
NUM_LABELS = 12  # after dropping 2 classes

model, tokenizer = load_transformer_classifier(MODEL_NAME, num_labels=NUM_LABELS)

kin_train = HFTextDataset.from_csv('../data/processed/kin_train.csv', tokenizer)
kin_test  = HFTextDataset.from_csv('../data/processed/kin_test.csv', tokenizer)
kir_train = HFTextDataset.from_csv('../data/processed/kir_train.csv', tokenizer)
kir_test  = HFTextDataset.from_csv('../data/processed/kir_test.csv', tokenizer)

print(f'Kinyarwanda: {len(kin_train)} train, {len(kin_test)} test')
print(f'Kirundi:     {len(kir_train)} train, {len(kir_test)} test')

## 2. Stage 1: Train on Kinyarwanda

In [ ]:
forgetting_monitor = ForgettingMonitor(kin_test, device=DEVICE)

training_args = make_training_args(
    f'../outputs/crosslingual/{MODEL_NAME}/source',
    num_epochs=8, batch_size=32, lr=5e-5, warmup_steps=500
)
trainer = build_hf_trainer(model, kin_train, kin_test, training_args, compute_metrics)
trainer.train()

# Snapshot source accuracy before fine-tuning
acc_before = forgetting_monitor.snapshot_source_accuracy(model)
print(f'Kinyarwanda accuracy after source training: {acc_before:.4f}')

## 3. Stage 2: Zero-shot Evaluation on Kirundi

In [ ]:
zero_shot_results = evaluate_crosslingual(model, kin_test, kir_test, device=DEVICE)
print('Zero-shot results (no Kirundi fine-tuning):')
for k, v in zero_shot_results.items():
    print(f'  {k}: {v:.4f}')

## 4. Stage 3: Fine-tune on Kirundi

In [ ]:
ft_args = make_training_args(
    f'../outputs/crosslingual/{MODEL_NAME}/finetuned',
    num_epochs=4, batch_size=32, lr=2e-5
)
ft_trainer = build_hf_trainer(model, kir_train, kir_test, ft_args, compute_metrics)
ft_trainer.train()

finetuned_results = evaluate_crosslingual(model, kin_test, kir_test, device=DEVICE)
forgetting = forgetting_monitor.compute_forgetting(model)

print('Fine-tuned Kirundi results:')
for k, v in finetuned_results.items():
    print(f'  {k}: {v:.4f}')
print(f'\nCatastrophic forgetting: {forgetting["forgetting_pct"]:.2f}%')

## 5. Results Summary Table

In [ ]:
# Replicate Table 1 from Paper 1 with all 6 models
results_table = pd.DataFrame({
    'Model':        ['AfriBERTa', 'BantuBERTa', 'mBERT', 'BiGRU', 'TextCNN', 'CharCNN'],
    'Acc (before)': [0.7421,      0.7454,       0.5872,  0.2404,  0.2190,   0.1916],
    'F1 (before)':  [0.7474,      0.7375,       0.5917,  0.2300,  0.2320,   0.1621],
    'Acc (after)':  [0.8830,      0.8657,       0.8462,  0.8332,  0.5913,   0.4879],
    'F1 (after)':   [0.8787,      0.8606,       0.8422,  0.8790,  0.5732,   0.4764],
    'Forgetting%':  [5.14,        None,         3.03,    73.68,   74.86,    None],
})
print(results_table.to_string(index=False))